# Evaluation Results Viewer

Load and explore evaluation results from `results/all_results.jsonl`

In [1]:
import pandas as pd
import json
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [2]:
RESULTS_FILE = Path("/home/hltcoe/rjha/rjha_exp/pylate-xtr/results/all_results.jsonl")

def load_results(filepath: Path = RESULTS_FILE) -> pd.DataFrame:
    """Load JSONL results and flatten nested structures."""
    records = []
    with open(filepath, 'r') as f:
        for line in f:
            data = json.loads(line)
            
            # Flatten all fields
            flat = {
                'dataset': data.get('dataset'),
                'model': data.get('model'),
                'model_dtype': data.get('model_dtype'),
                'embedding_dtype': data.get('embedding_dtype'),
                'query_length': data.get('query_length'),
                'doc_length': data.get('doc_length'),
                'lowercase': data.get('lowercase'),
                'k': data.get('k'),
                'k_token': data.get('k_token'),
                'retrieval_mode': data.get('retrieval_mode'),
                'index_time': data.get('index_time'),
                'retrieve_time': data.get('retrieve_time'),
                'encode_batch_size': data.get('encode_batch_size'),
                'retrieval_batch_size': data.get('retrieval_batch_size'),
                'timestamp': data.get('timestamp'),
                'run_id': data.get('run_id'),
            }
            
            # Add evaluation scores
            scores = data.get('evaluation_scores', {})
            flat['map'] = scores.get('map')
            flat['ndcg@10'] = scores.get('ndcg@10')
            flat['ndcg@100'] = scores.get('ndcg@100')
            flat['recall@10'] = scores.get('recall@10')
            flat['recall@100'] = scores.get('recall@100')
            flat['hit_rate@5'] = scores.get('hit_rate@5')
            
            # Add imputation info
            imputation = data.get('imputation', {})
            flat['imputation_method'] = imputation.get('method') if imputation else "min"
            flat['imputation_percentile'] = imputation.get('percentile') if imputation else None
            flat['power_law_multiplier'] = imputation.get('power_law_multiplier') if imputation else None
            
            # Add index config info
            idx_cfg = data.get('index_config', {})
            flat['index_type'] = idx_cfg.get('name')
            
            records.append(flat)
    
    df = pd.DataFrame(records)
    
    # Apply default values for missing columns (only for non-None defaults)
    defaults = {
        'retrieval_mode': 'ColBERT',  # Older runs were ColBERT by default
    }
    for col, default in defaults.items():
        if col in df.columns:
            df[col] = df[col].fillna(default)
    
    return df

df = load_results()
print(f"Loaded {len(df)} results")

Loaded 1657 results


In [3]:
def parse_dataset_name(dataset: str) -> str:
    """Extract readable dataset name from full path."""
    parts = dataset.split('/')
    if parts[0] == 'beir':
        # beir/nfcorpus/test -> nfcorpus, beir/trec-covid -> trec-covid
        return parts[1]
    elif parts[0] == 'lotte':
        # lotte/lifestyle/dev/search -> lotte-lifestyle
        return f"lotte-{parts[1]}"
    elif parts[0] == 'nano-beir':
        # nano-beir/msmarco -> nano-msmarco
        return f"nano-{parts[1]}"
    else:
        return dataset

def parse_model_name(model: str) -> str:
    """Extract model name, treating checkpoint-X and final as the same model."""
    p = Path(model)
    # Both 'checkpoint-XXXXX' and 'final' are subdirs of the actual model dir
    if 'checkpoint' in p.name or p.name == 'final':
        return p.parent.name
    return p.name

# Extract short names for readability
df['model_short'] = df['model'].apply(parse_model_name)
df['dataset_short'] = df['dataset'].apply(parse_dataset_name)

In [11]:
# Default filters - used when not specified in filter()
DEFAULT_MODELS = [
    'experiment_1_contrastive_colbert_bs196_50k',
    'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k',
    'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix',
]

def filter_df(
    models: list | None = DEFAULT_MODELS,
    retrieval_modes: list | None = None,
    imputation_methods: list | None = None,
    datasets: list | None = None,
    k_tokens: list | None = None,
) -> pd.DataFrame:
    """Filter dataframe with defaults. Pass None to include all for that dimension."""
    result = df.copy()
    if models is not None:
        result = result[result['model_short'].isin(models)]
    if retrieval_modes is not None:
        result = result[result['retrieval_mode'].isin(retrieval_modes)]
    if imputation_methods is not None:
        result = result[result['imputation_method'].isin(imputation_methods)]
    if datasets is not None:
        result = result[result['dataset_short'].isin(datasets)]
    if k_tokens is not None:
        result = result[result['k_token'].isin(k_tokens)]
    return result

print(f"Total results: {len(df)}")
print(f"With default filter: {len(filter_df())}")

Total results: 1657
With default filter: 109


## Overview

In [12]:
fdf = filter_df()
print("Unique models:")
for m in fdf['model_short'].unique():
    print(f"  - {m}")
print(f"\nUnique datasets: {list(fdf['dataset_short'].unique())}")
print(f"Index types: {list(fdf['index_type'].unique())}")
print(f"Retrieval modes: {list(fdf['retrieval_mode'].unique())}")
print(f"Imputation methods: {list(fdf['imputation_method'].dropna().unique())}")

Unique models:
  - experiment_1_contrastive_colbert_bs196_50k
  - experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k
  - experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix

Unique datasets: ['nfcorpus', 'fiqa', 'scidocs', 'scifact', 'trec-covid', 'nq', 'webis-touche2020', 'quora', 'lotte-lifestyle']
Index types: ['ScaNN']
Retrieval modes: ['ColBERT', 'XTR']
Imputation methods: ['min', 'percentile', 'mean', 'power_law']


## Results Table

Key metrics formatted for readability (scores as percentages)

In [13]:
# Display key columns with formatted scores
fdf = filter_df(imputation_methods=['min'])  # Default to min imputation for clean comparison

display_cols = ['dataset_short', 'model_short', 'retrieval_mode', 'imputation_method', 
                'index_type', 'ndcg@10', 'ndcg@100', 'recall@100', 'map']
results_display = fdf[display_cols].copy()

# Format numeric columns as percentages
score_cols = ['ndcg@10', 'ndcg@100', 'recall@100', 'map']
for col in score_cols:
    results_display[col] = results_display[col].apply(lambda x: f"{x*100:.2f}" if pd.notna(x) else "")

results_display.sort_values(['dataset_short', 'model_short', 'retrieval_mode'])

,dataset_short,model_short,retrieval_mode,imputation_method,index_type,ndcg@10,ndcg@100,recall@100,map
1549,fiqa,experiment_1_contrastive_colbert_bs196_50k,ColBERT,min,ScaNN,37.15,43.02,65.60,31.25
1616,fiqa,experiment_1_contrastive_colbert_bs196_50k,XTR,min,ScaNN,36.84,42.60,65.40,30.73
1554,fiqa,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,36.87,43.19,66.06,31.53
1632,fiqa,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,34.09,40.30,63.46,28.71
1651,fiqa,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,34.18,40.38,63.69,28.74
...,...,...,...,...,...,...,...,...,...
1637,trec-covid,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,78.14,56.23,13.38,10.54
1656,trec-covid,experiment_1_contrastive_xtr_primeqa_kprime_12...,XTR,min,ScaNN,77.09,56.24,13.39,10.55
1618,webis-touche2020,experiment_1_contrastive_colbert_bs196_50k,ColBERT,min,ScaNN,23.43,35.47,47.11,15.24
1619,webis-touche2020,experiment_1_contrastive_colbert_bs196_50k,ColBERT,min,ScaNN,23.43,35.47,47.11,15.24


## Pivot Table: NDCG@10 by Dataset

In [14]:
# Pivot table: NDCG@10 by dataset and model
fdf = filter_df(imputation_methods=['min'])

pivot = fdf.pivot_table(
    index='dataset_short', 
    columns=['model_short', 'retrieval_mode'], 
    values='ndcg@10',
    aggfunc='max'
) * 100

pivot.round(2).style.background_gradient(cmap='YlGn', axis="columns").format("{:.2f}").set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'right')]}
])

## Compare Retrieval Modes

In [15]:
# Compare ColBERT vs XTR retrieval modes
fdf = filter_df(imputation_methods=['min'])

mode_comparison = fdf.pivot_table(
    index=['dataset_short', 'model_short'],
    columns='retrieval_mode',
    values='ndcg@10',
    aggfunc='max'
) * 100

mode_comparison.round(2)

retrieval_mode                                                       ColBERT  \
dataset_short    model_short                                                   
fiqa             experiment_1_contrastive_colbert_bs196_50k            37.15   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
lotte-lifestyle  experiment_1_contrastive_colbert_bs196_50k            49.32   
nfcorpus         experiment_1_contrastive_colbert_bs196_50k            34.49   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
nq               experiment_1_contrastive_colbert_bs196_50k            53.07   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
quora            experiment_1_contrastive_colbert_bs196_50k            87.50   
scidocs          experiment_1_contrastive_colbert_bs196_50k            18.54   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
scifact          experiment_1_contrastive_colbert_bs196_50k            70.59   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
trec-covid       experiment_1_contrastive_colbert_bs196_50k            78.45   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
                 experiment_1_contrastive_xtr_primeqa_kprime_128...      NaN   
webis-touche2020 experiment_1_contrastive_colbert_bs196_50k            23.43   

retrieval_mode                                                         XTR  
dataset_short    model_short                                                
fiqa             experiment_1_contrastive_colbert_bs196_50k          36.84  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  36.87  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  34.18  
lotte-lifestyle  experiment_1_contrastive_colbert_bs196_50k          49.26  
nfcorpus         experiment_1_contrastive_colbert_bs196_50k          34.38  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  35.00  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  33.12  
nq               experiment_1_contrastive_colbert_bs196_50k          52.80  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  51.72  
quora            experiment_1_contrastive_colbert_bs196_50k          87.41  
scidocs          experiment_1_contrastive_colbert_bs196_50k          18.54  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  17.07  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  16.93  
scifact          experiment_1_contrastive_colbert_bs196_50k          69.88  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  66.18  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  64.15  
trec-covid       experiment_1_contrastive_colbert_bs196_50k          78.23  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  78.93  
                 experiment_1_contrastive_xtr_primeqa_kprime_128...  78.14  
webis-touche2020 experiment_1_contrastive_colbert_bs196_50k          23.20

## Compare Imputation Methods (XTR)

In [ ]:
# Compare imputation methods across all datasets for XTR model
XTR_MODEL = 'experiment_1_contrastive_xtr_primeqa_kprime_128_bs196_50k_bugfix'
COLBERT_MODEL = 'experiment_1_contrastive_colbert_bs196_50k'

xtr_df = filter_df(
    models=[XTR_MODEL],
    retrieval_modes=['XTR'],
    datasets=['nfcorpus', 'fiqa', 'trec-covid', 'nq'],
    k_tokens=[10_000, 40_000],
)

if len(xtr_df) > 0:
    imputation_pivot = xtr_df.pivot_table(
        index=['dataset_short', 'k_token'],
        columns=['imputation_method'],
        values='ndcg@10',
        aggfunc='max'
    ) * 100
    
    styled = imputation_pivot.round(2).style.format("{:.2f}").set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'right')]}
    ])
    display(styled)
else:
    print("No XTR runs found for this model")

## Filter and Explore

In [ ]:
# Example: Filter by dataset
# dataset_filter = 'fiqa'
# df[df['dataset_short'].str.contains(dataset_filter, case=False)][display_cols]

In [ ]:
# Example: Get best run per dataset
# df.loc[df.groupby('dataset_short')['ndcg@10'].idxmax()][display_cols]

## Timing Information

In [ ]:
fdf = filter_df(imputation_methods=['min'])

timing_cols = ['dataset_short', 'model_short', 'retrieval_mode', 'index_type', 'index_time', 'retrieve_time']
timing_df = fdf[timing_cols].copy()
timing_df['index_time'] = timing_df['index_time'].apply(lambda x: f"{x:.1f}s" if pd.notna(x) else "")
timing_df['retrieve_time'] = timing_df['retrieve_time'].apply(lambda x: f"{x:.1f}s" if pd.notna(x) else "")
timing_df.sort_values(['dataset_short', 'model_short'])

## Raw DataFrame

In [ ]:
# Access full dataframe for custom analysis
df.head()